# Handling Late Data and Watermarking

Experience how late arrivals and output modes interleaves in windowed aggregation queries.

**NOTE**: Run the other notebook first.

In [1]:
import os
from pyspark.sql import SparkSession
from pyspark.streaming import StreamingContext
import io
from pyspark.sql.functions import *
import time
import json
import struct
import requests 

import glob, pyspark

# the Kafka connector is not part of Spark: it is fetched from Maven, and its
# coordinate has to match *this* Spark and the Scala it was built against. Derive
# both instead of hard-coding them, so the notebook cannot disagree with its image.
jars  = glob.glob(pyspark.__path__[0] + '/jars/scala-library-2.13*')
scala = '2.13' if jars else '2.12'
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages org.apache.spark:spark-sql-kafka-0-10_%s:%s pyspark-shell'
    % (scala, pyspark.__version__))
print('spark', pyspark.__version__, '/ scala', scala)
                                    
spark = (SparkSession.builder 
    .master("local[*]")
    .appName("test")
    .getOrCreate()
        )

spark

spark 3.5.0 / scala 2.12


set up the environment variables

In [2]:
servers = "kafka:9092"
topic = "words"

In [3]:
from pyspark.sql.types import *

schema = StructType([
    StructField("word", StringType(), True),
    StructField("ts", TimestampType(), True)])

In [4]:
raw_sdf = (spark
  .readStream
  .format("kafka")
  .option("kafka.bootstrap.servers", servers)
  .option("startingOffsets", "latest")
  .option("subscribe", topic)
  .load())

In [5]:
raw_sdf.isStreaming

True

In [6]:
words=(raw_sdf.select(from_json(col("value").cast("string"), schema).alias("value"))
              .select("value.*"))

In [7]:
words.printSchema()

root
 |-- word: string (nullable = true)
 |-- ts: timestamp (nullable = true)



In [8]:
q_update=(words.withWatermark("ts","10 minutes").groupBy(window(words.ts, "10 minutes", "5 minutes"),words.word).count()
   .writeStream
   .format("memory")
   .outputMode("update") 
   .queryName("sinkTable_update")
   .start())

In [9]:
q_append=(words.withWatermark("ts","10 minutes").groupBy(window("ts", "10 minutes", "5 minutes"),words.word).count()
   .writeStream
   .format("memory")
   .outputMode("append") 
   .queryName("sinkTable_append")
   .start())

Go the the other notebook and run the cells in **Section 1)**

### Why the two `status` cells, and what to wait for

Spark consumes on its own schedule, not on yours. The two cells below ask each query
what it is doing right now, and you read the result **before** looking at the sink
tables: a query still busy has not finished folding in what the simulator just sent,
so the tables would show you a half-processed picture — and you would read a timing
accident as a difference in semantics.

Wait until **both** queries report exactly this:

```
{'message': 'Waiting for data to arrive',
 'isDataAvailable': False,
 'isTriggerActive': False}
```

`isDataAvailable: False` means there is nothing left unread on the topic, and
`isTriggerActive: False` means no micro-batch is running. Together they say: everything
produced so far has been processed, and the result tables are stable. If instead you see
`'Processing new data'` or `isTriggerActive: True`, re-run the cell — it takes a second
or two.

In [11]:
q_update.status

{'message': 'Waiting for data to arrive',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [12]:
q_append.status

{'message': 'Waiting for data to arrive',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [13]:
# look up the most recent results
spark.sql("SELECT * FROM sinkTable_update ORDER BY window,word").show(15,False) # without ORDER BY TS DESC because the result in the table is already only the most recent

+------------------------------------------+----+-----+
|window                                    |word|count|
+------------------------------------------+----+-----+
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|dog |1    |
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|owl |1    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|dog |1    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|owl |1    |
+------------------------------------------+----+-----+



In [14]:
# look up the most recent results
spark.sql("SELECT * FROM sinkTable_append ORDER BY window,word").show(10,False) # without ORDER BY TS DESC because the result in the table is already only the most recent

+------+----+-----+
|window|word|count|
+------+----+-----+
+------+----+-----+



Go the the other notebook and run the cells in **Section 2)**

In [15]:
q_update.status

{'message': 'Waiting for data to arrive',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [16]:
q_append.status

{'message': 'Waiting for data to arrive',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [17]:
# look up the most recent results
spark.sql("SELECT * FROM sinkTable_update ORDER BY window,word,count").show(15,False) # without ORDER BY TS DESC because the result in the table is already only the most recent

+------------------------------------------+----+-----+
|window                                    |word|count|
+------------------------------------------+----+-----+
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|cat |1    |
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|dog |1    |
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|owl |1    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|cat |1    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|dog |1    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|dog |2    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|owl |1    |
|{2024-03-24 12:10:00, 2024-03-24 12:20:00}|dog |1    |
+------------------------------------------+----+-----+



In [18]:
# look up the most recent results
spark.sql("SELECT * FROM sinkTable_append ORDER BY window,word").show(10,False) # without ORDER BY TS DESC because the result in the table is already only the most recent

+------+----+-----+
|window|word|count|
+------+----+-----+
+------+----+-----+



Go the the other notebook and run the cells in **Section 3)**

In [21]:
q_update.status

{'message': 'Waiting for data to arrive',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [22]:
q_append.status

{'message': 'Waiting for data to arrive',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [23]:
# look up the most recent results
spark.sql("SELECT * FROM sinkTable_update ORDER BY window,word,count").show(15,False) # without ORDER BY TS DESC because the result in the table is already only the most recent

+------------------------------------------+----+-----+
|window                                    |word|count|
+------------------------------------------+----+-----+
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|cat |1    |
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|dog |1    |
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|dog |2    |
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|owl |1    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|cat |1    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|dog |1    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|dog |2    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|dog |3    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|owl |1    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|owl |2    |
|{2024-03-24 12:10:00, 2024-03-24 12:20:00}|cat |1    |
|{2024-03-24 12:10:00, 2024-03-24 12:20:00}|dog |1    |
|{2024-03-24 12:10:00, 2024-03-24 12:20:00}|owl |1    |
|{2024-03-24 12:15:00, 2024-03-24 12:25:00}|cat |1    |
+------------------------------------------+----

In [24]:
# look up the most recent results
spark.sql("SELECT * FROM sinkTable_append ORDER BY window,word").show(10,False) # without ORDER BY TS DESC because the result in the table is already only the most recent

+------+----+-----+
|window|word|count|
+------+----+-----+
+------+----+-----+



Go the the other notebook and run the cells in **Section 4)**

In [25]:
q_update.status

{'message': 'Waiting for data to arrive',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [26]:
q_append.status

{'message': 'Waiting for data to arrive',
 'isDataAvailable': False,
 'isTriggerActive': False}

In [27]:
# look up the most recent results
spark.sql("SELECT * FROM sinkTable_update ORDER BY window,word,count").show(15,False) # without ORDER BY TS DESC because the result in the table is already only the most recent

+------------------------------------------+------+-----+
|window                                    |word  |count|
+------------------------------------------+------+-----+
|{2024-03-24 11:55:00, 2024-03-24 12:05:00}|donkey|1    |
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|cat   |1    |
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|dog   |1    |
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|dog   |2    |
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|donkey|1    |
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|owl   |1    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|cat   |1    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|dog   |1    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|dog   |2    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|dog   |3    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|owl   |1    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|owl   |2    |
|{2024-03-24 12:10:00, 2024-03-24 12:20:00}|cat   |1    |
|{2024-03-24 12:10:00, 2024-03-24 12:20:00}|dog   |1    |
|{2024-03-24 1

In [28]:
# look up the most recent results
spark.sql("SELECT * FROM sinkTable_append ORDER BY window,word").show(10,False) # without ORDER BY TS DESC because the result in the table is already only the most recent

+------------------------------------------+------+-----+
|window                                    |word  |count|
+------------------------------------------+------+-----+
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|cat   |1    |
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|dog   |2    |
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|donkey|1    |
|{2024-03-24 12:00:00, 2024-03-24 12:10:00}|owl   |1    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|cat   |1    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|dog   |3    |
|{2024-03-24 12:05:00, 2024-03-24 12:15:00}|owl   |2    |
+------------------------------------------+------+-----+



Too late, or not?

https://spark.apache.org/docs/latest/structured-streaming-programming-guide.html#semantic-guarantees-of-aggregation-with-watermarking

---
## Clean up

Stop both queries. They are still running: a streaming query keeps its state and its
Kafka consumer alive until you stop it or the kernel dies, and two abandoned queries are
how the next run of this notebook ends up reading its own leftovers.

The topic itself is deleted by the last cell of the **simulator** notebook.

In [29]:
for q in (q_update, q_append):
    q.stop()

print("update:", q_update.isActive, " append:", q_append.isActive)

update: False  append: False
